# Analise do inventario em Parquet

Notebook para ler o arquivo gerado por:

```bash
python main.py export --format parquet
```

Para analise local exploratoria, este notebook usa DuckDB como engine principal. Ele consulta Parquet direto no disco, sem precisar carregar tudo em memoria.

## Qual engine usar?

- **DuckDB**: melhor escolha inicial para explorar Parquet local, fazer SQL, agregacoes e filtros rapidamente.
- **Pandas**: bom para amostras pequenas, graficos e manipulacoes simples em memoria.
- **PySpark**: melhor quando o volume for grande demais para a maquina, quando houver muitos Parquets particionados, ou quando a analise virar pipeline distribuido.

Recomendacao para este projeto agora: **DuckDB primeiro**. Use PySpark depois se o dado crescer muito ou se voce for processar tudo em cluster.

In [15]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PARQUET_PATH = PROJECT_ROOT / "exports" / "inventory.parquet"

print("Projeto:", PROJECT_ROOT)
print("Parquet:", PARQUET_PATH)
print("Existe:", PARQUET_PATH.exists())

if not PARQUET_PATH.exists():
    raise FileNotFoundError(
        f"Parquet nao encontrado: {PARQUET_PATH}. "
        "Gere o arquivo com: python main.py export --format parquet"
    )

Projeto: /Users/analuizadantas/Library/CloudStorage/OneDrive-Fiap-FaculdadedeInformáticaeAdministraçãoPaulista/Codigos/Projeto_sharepoint
Parquet: /Users/analuizadantas/Library/CloudStorage/OneDrive-Fiap-FaculdadedeInformáticaeAdministraçãoPaulista/Codigos/Projeto_sharepoint/exports/inventory.parquet
Existe: True


## Instalar dependencias, se necessario

Execute a celula abaixo somente se `import duckdb` falhar.

In [ ]:
#%pip install duckdb pandas pyarrow

  Using cached duckdb-1.5.3-cp313-cp313-macosx_10_13_x86_64.whl.metadata (4.2 kB)
Using cached duckdb-1.5.3-cp313-cp313-macosx_10_13_x86_64.whl (17.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 14.1 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 44.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pandas]2m2/3 [pandas]
Note: you may need to restart the kernel to use updated packages.


## Abrir DuckDB e criar view sobre o Parquet

In [16]:
import duckdb

parquet_sql_path = str(PARQUET_PATH).replace("'", "''")
con = duckdb.connect(database=":memory:")
con.execute(f"CREATE OR REPLACE VIEW inventory AS SELECT * FROM read_parquet('{parquet_sql_path}')")

con.sql("DESCRIBE inventory").df()

,column_name,column_type,null,key,default,extra
0,tenant_site,VARCHAR,YES,None,None,None
1,site_name,VARCHAR,YES,None,None,None
2,site_url,VARCHAR,YES,None,None,None
3,biblioteca,VARCHAR,YES,None,None,None
4,caminho_completo_da_pasta,VARCHAR,YES,None,None,None
5,nome_arquivo_ou_pasta,VARCHAR,YES,None,None,None
6,tipo_item,VARCHAR,YES,None,None,None
7,extensao,VARCHAR,YES,None,None,None
8,tipo_mime,VARCHAR,YES,None,None,None
9,tamanho_bytes,BIGINT,YES,None,None,None


## Sites presentes no Parquet

Lista todos os sites encontrados no arquivo Parquet, com contagem de bibliotecas, arquivos, pastas e volume total de arquivos.

In [17]:
sites_df = con.sql("""
SELECT
    site_name,
    site_url,
    tenant_site,
    COUNT(DISTINCT biblioteca) AS bibliotecas,
    COUNT_IF(tipo_item = 'file') AS arquivos,
    COUNT_IF(tipo_item = 'folder') AS pastas,
    SUM(CASE WHEN tipo_item = 'file' THEN tamanho_bytes ELSE 0 END) AS total_bytes_arquivos,
    ROUND(SUM(CASE WHEN tipo_item = 'file' THEN tamanho_bytes ELSE 0 END) / 1024.0 / 1024.0 / 1024.0, 2) AS total_gb_arquivos
FROM inventory
GROUP BY site_name, site_url, tenant_site
ORDER BY total_bytes_arquivos DESC NULLS LAST, site_name
""").df()

sites_df

,site_name,site_url,tenant_site,bibliotecas,arquivos,pastas,total_bytes_arquivos,total_gb_arquivos
0,Expanzio,https://expanziobr.sharepoint.com/sites/Expanzio,expanziobr.sharepoint.com,1,368317.0,141898.0,1.175054e+12,1094.35
1,Marketing,https://expanziobr.sharepoint.com/sites/Marketing,expanziobr.sharepoint.com,1,5097.0,274.0,3.105802e+11,289.25
2,Comercial,https://expanziobr.sharepoint.com/sites/Comercial,expanziobr.sharepoint.com,1,9940.0,462.0,2.137939e+10,19.91
3,Financeiro,https://expanziobr.sharepoint.com/sites/Financ...,expanziobr.sharepoint.com,1,7353.0,653.0,1.644318e+09,1.53
4,Licenciamento,https://expanziobr.sharepoint.com/sites/Licenc...,expanziobr.sharepoint.com,1,68.0,8.0,2.187074e+07,0.02
5,RH,https://expanziobr.sharepoint.com/sites/RH,expanziobr.sharepoint.com,1,51.0,18.0,1.483471e+07,0.01
6,"expanziobr-my.sharepoint.com,12b045b6-65ad-459...",https://expanziobr-my.sharepoint.com,expanziobr-my.sharepoint.com,2,240.0,3.0,8.452940e+05,0.00
7,Tenant Administration,https://expanziobr-admin.sharepoint.com,expanziobr-admin.sharepoint.com,4,1.0,5.0,2.880000e+02,0.00
8,Aplicativos,https://expanziobr.sharepoint.com/sites/appcat...,expanziobr.sharepoint.com,3,0.0,3.0,0.000000e+00,0.00
9,Comercial-Propostas Comerciais,https://expanziobr.sharepoint.com/sites/Comerc...,expanziobr.sharepoint.com,1,0.0,1.0,0.000000e+00,0.00


## Contagens e qualidade basica

In [11]:
con.sql("""
SELECT
    COUNT(*) AS total_linhas,
    COUNT_IF(tipo_item = 'file') AS arquivos,
    COUNT_IF(tipo_item = 'folder') AS pastas,
    COUNT(DISTINCT site_url) AS sites,
    COUNT(DISTINCT id_drive) AS drives,
    COUNT(DISTINCT biblioteca) AS bibliotecas,
    SUM(CASE WHEN tipo_item = 'file' THEN tamanho_bytes ELSE 0 END) AS total_bytes_arquivos,
    MIN(data_modificacao) AS menor_data_modificacao,
    MAX(data_modificacao) AS maior_data_modificacao
FROM inventory
""").df()

,total_linhas,arquivos,pastas,sites,drives,bibliotecas,total_bytes_arquivos,menor_data_modificacao,maior_data_modificacao
0,520617,378257.0,142360.0,2,2,1,1.196433e+12,1983-11-17T03:55:42Z,2026-06-04T03:31:20Z


In [12]:
con.sql("""
SELECT
    tipo_item,
    status_leitura,
    COUNT(*) AS quantidade
FROM inventory
GROUP BY tipo_item, status_leitura
ORDER BY tipo_item, status_leitura
""").df()

,tipo_item,status_leitura,quantidade
0,file,ok,378257
1,folder,ok,142360


## Amostra dos metadados

In [13]:
con.sql("""
SELECT
    site_name,
    biblioteca,
    caminho_item,
    nome_arquivo_ou_pasta,
    tipo_item,
    extensao,
    tipo_mime,
    tamanho_bytes,
    data_criacao,
    data_modificacao,
    data_ultimo_uso_acesso,
    caminho_completo_da_pasta
FROM inventory
WHERE tipo_item = 'file'
ORDER BY data_modificacao DESC NULLS LAST
LIMIT 30
""").df()

,site_name,biblioteca,caminho_item,nome_arquivo_ou_pasta,tipo_item,extensao,tipo_mime,tamanho_bytes,data_criacao,data_modificacao,data_ultimo_uso_acesso,caminho_completo_da_pasta
0,Expanzio,Documents,Documents/General/1. Corporativos/GRUPO BOTICÁ...,LJ SHOP. METRÔ BOULEVARD - BOT. - LEV. ARQ. - ...,file,pdf,application/pdf,506905,2026-06-04T03:30:46Z,2026-06-04T03:30:46Z,<NA>,Documents/General/1. Corporativos/GRUPO BOTICÁ...
1,Expanzio,Documents,Documents/General/1. Corporativos/GRUPO BOTICÁ...,LJ SHOP. METRÔ BOULEVARD - BOT. - LEV. ARQ. - ...,file,pdf,application/pdf,509081,2026-06-04T03:30:46Z,2026-06-04T03:30:46Z,<NA>,Documents/General/1. Corporativos/GRUPO BOTICÁ...
2,Expanzio,Documents,Documents/General/1. Corporativos/GRUPO BOTICÁ...,LJ SHOP. METRÔ BOULEVARD - BOT. - LEV. ARQ. - ...,file,pdf,application/pdf,509691,2026-06-04T03:30:46Z,2026-06-04T03:30:46Z,<NA>,Documents/General/1. Corporativos/GRUPO BOTICÁ...
3,Expanzio,Documents,Documents/General/1. Corporativos/GRUPO BOTICÁ...,LJ SHOP. METRÔ BOULEVARD - BOT. - LEV. ARQ. - ...,file,pdf,application/pdf,499966,2026-06-04T03:30:46Z,2026-06-04T03:30:46Z,<NA>,Documents/General/1. Corporativos/GRUPO BOTICÁ...
4,Expanzio,Documents,Documents/General/1. Corporativos/GRUPO BOTICÁ...,LJ SHOP. METRÔ BOULEVARD - BOT. - LEV. ARQ.- R...,file,dwg,application/octet-stream,16136189,2026-06-04T03:30:34Z,2026-06-04T03:30:34Z,<NA>,Documents/General/1. Corporativos/GRUPO BOTICÁ...
5,Expanzio,Documents,Documents/General/1. Corporativos/SMART FIT/RN...,SBRRNISGA02-EXE-INC-000-PROJ-R02.dwl,file,dwl,application/octet-stream,68,2026-06-03T23:12:15Z,2026-06-03T23:12:07Z,<NA>,Documents/General/1. Corporativos/SMART FIT/RN...
6,Expanzio,Documents,Documents/General/1. Corporativos/SMART FIT/RN...,SBRRNISGA02-EXE-INC-000-PROJ-R02.dwl2,file,dwl2,application/octet-stream,218,2026-06-03T23:12:15Z,2026-06-03T23:12:07Z,<NA>,Documents/General/1. Corporativos/SMART FIT/RN...
7,Expanzio,Documents,Documents/General/1. Corporativos/BRADESCO/SP/...,AV PAULISTA BELA VISTA-BRADESCO-EVTL-R00.pdf,file,pdf,application/pdf,13943379,2026-06-03T22:10:04Z,2026-06-03T22:10:04Z,<NA>,Documents/General/1. Corporativos/BRADESCO/SP/...
8,Expanzio,Documents,Documents/General/1. Corporativos/BACCARAT BRA...,BACCARAT-PROJ-INC-R00-002.pdf,file,pdf,application/pdf,279023,2026-06-03T21:15:04Z,2026-06-03T21:18:10Z,<NA>,Documents/General/1. Corporativos/BACCARAT BRA...
9,Expanzio,Documents,Documents/General/1. Corporativos/BACCARAT BRA...,BACCARAT-PROJ-INC-R00-001.pdf,file,pdf,application/pdf,335186,2026-06-03T21:15:06Z,2026-06-03T21:17:45Z,<NA>,Documents/General/1. Corporativos/BACCARAT BRA...


## Extensoes mais comuns

In [14]:
con.sql("""
SELECT
    COALESCE(NULLIF(extensao, ''), '(sem extensao)') AS extensao,
    COUNT(*) AS arquivos,
    SUM(tamanho_bytes) AS total_bytes,
    ROUND(SUM(tamanho_bytes) / 1024.0 / 1024.0 / 1024.0, 2) AS total_gb
FROM inventory
WHERE tipo_item = 'file'
GROUP BY 1
ORDER BY arquivos DESC
LIMIT 50
""").df()

,extensao,arquivos,total_bytes,total_gb
0,pdf,170157,2.188267e+11,203.80
1,dwg,76742,2.094132e+11,195.03
2,jpg,50875,1.055206e+11,98.27
3,jpeg,17488,7.081516e+09,6.60
4,docx,11640,2.051431e+10,19.11
5,bak,7192,9.081917e+10,84.58
6,zip,6071,1.972960e+11,183.75
7,png,5880,8.542722e+09,7.96
8,rfa,4464,2.975651e+09,2.77
9,xlsx,3158,8.195596e+08,0.76


## Bibliotecas por volume

In [12]:
con.sql("""
SELECT
    site_name,
    site_url,
    biblioteca,
    COUNT(*) AS arquivos,
    SUM(tamanho_bytes) AS total_bytes,
    ROUND(SUM(tamanho_bytes) / 1024.0 / 1024.0 / 1024.0, 2) AS total_gb
FROM inventory
WHERE tipo_item = 'file'
GROUP BY site_name, site_url, biblioteca
ORDER BY total_bytes DESC
LIMIT 50
""").df()

,site_name,site_url,biblioteca,arquivos,total_bytes,total_gb
0,Comercial,https://expanziobr.sharepoint.com/sites/Comercial,Documents,9940,2.137939e+10,19.91


## Pastas por volume calculado

In [13]:
con.sql("""
SELECT
    site_name,
    site_url,
    biblioteca,
    caminho_item AS pasta,
    quantidade_arquivos_dentro_da_pasta AS arquivos_na_pasta,
    volume_total_pasta_bytes,
    ROUND(volume_total_pasta_bytes / 1024.0 / 1024.0 / 1024.0, 2) AS total_gb,
    data_modificacao
FROM inventory
WHERE tipo_item = 'folder'
ORDER BY volume_total_pasta_bytes DESC NULLS LAST
LIMIT 50
""").df()

,site_name,site_url,biblioteca,pasta,arquivos_na_pasta,volume_total_pasta_bytes,total_gb,data_modificacao
0,Comercial,https://expanziobr.sharepoint.com/sites/Comercial,Documents,root/General,9940,21379385513,19.91,2025-07-19T09:30:57Z
1,Comercial,https://expanziobr.sharepoint.com/sites/Comercial,Documents,root,9940,21379385513,19.91,2026-06-01T18:05:02Z
2,Comercial,https://expanziobr.sharepoint.com/sites/Comercial,Documents,root/General/Pastas q não pertecem ao comercial,801,9152667801,8.52,2025-10-11T13:21:46Z
3,Comercial,https://expanziobr.sharepoint.com/sites/Comercial,Documents,root/General/Pastas q não pertecem ao comercia...,33,8322764687,7.75,2025-10-11T13:45:20Z
4,Comercial,https://expanziobr.sharepoint.com/sites/Comercial,Documents,root/General/Pastas q não pertecem ao comercia...,25,8322251027,7.75,2025-07-07T20:24:47Z
5,Comercial,https://expanziobr.sharepoint.com/sites/Comercial,Documents,root/General/Pastas q não pertecem ao comercia...,7,8311518815,7.74,2025-11-27T19:56:38Z
6,Comercial,https://expanziobr.sharepoint.com/sites/Comercial,Documents,"root/General/Apresentações Comerciais, Meet e ...",60,6544691565,6.10,2024-10-01T18:12:33Z
7,Comercial,https://expanziobr.sharepoint.com/sites/Comercial,Documents,"root/General/Apresentações Comerciais, Meet e ...",7,3361998692,3.13,2023-03-21T15:18:16Z
8,Comercial,https://expanziobr.sharepoint.com/sites/Comercial,Documents,"root/General/Apresentações Comerciais, Meet e ...",2,2949184913,2.75,2023-04-05T15:33:26Z
9,Comercial,https://expanziobr.sharepoint.com/sites/Comercial,Documents,root/General/1. Operações e Desenvolvimento Co...,317,1973079346,1.84,2025-10-25T22:15:56Z


## Arquivos antigos ou sem ultimo acesso

In [14]:
con.sql("""
SELECT
    COUNT(*) AS arquivos,
    COUNT_IF(data_ultimo_uso_acesso IS NULL OR data_ultimo_uso_acesso = '') AS sem_ultimo_acesso,
    COUNT_IF(data_modificacao IS NULL OR data_modificacao = '') AS sem_data_modificacao,
    MIN(data_modificacao) AS menor_data_modificacao,
    MAX(data_modificacao) AS maior_data_modificacao
FROM inventory
WHERE tipo_item = 'file'
""").df()

,arquivos,sem_ultimo_acesso,sem_data_modificacao,menor_data_modificacao,maior_data_modificacao
0,9940,9940.0,0.0,2017-02-23T11:39:56Z,2026-06-01T18:05:02Z


## Opcional: usar Pandas em uma amostra

Use Pandas para graficos ou ajustes pequenos depois de filtrar/agregar com DuckDB.

In [15]:
sample_df = con.sql("""
SELECT *
FROM inventory
WHERE tipo_item = 'file'
ORDER BY data_modificacao DESC NULLS LAST
LIMIT 1000
""").df()

sample_df.head()

,tenant_site,site_name,site_url,biblioteca,caminho_completo_da_pasta,nome_arquivo_ou_pasta,tipo_item,extensao,tipo_mime,tamanho_bytes,...,data_modificacao,data_ultimo_uso_acesso,quantidade_arquivos_dentro_da_pasta,volume_total_pasta_bytes,volume_total_pasta_formatado,id_item,id_drive,status_leitura,data_hora_coleta,caminho_item
0,expanziobr.sharepoint.com,Comercial,https://expanziobr.sharepoint.com/sites/Comercial,Documents,root/General/9. Nossa LPU,LPU EXPANZIO_V3 (1).xlsx,file,xlsx,application/vnd.openxmlformats-officedocument....,192578,...,2026-06-01T18:05:02Z,<NA>,0,0,None,01NA37BFRGZCNY7EDCYJYJYVZZQZASPW26,b!FIB3Z0KRfU6tyPRnX3MFBomAIQiIsJtIhXnCtudAYp8r...,ok,2026-06-03T19:39:42+00:00,root/General/9. Nossa LPU/LPU EXPANZIO_V3 (1)....
1,expanziobr.sharepoint.com,Comercial,https://expanziobr.sharepoint.com/sites/Comercial,Documents,root/General/7. Credenciamento LPU-BID-RFQ,LPU_Mercado Livte.xlsx,file,xlsx,application/vnd.openxmlformats-officedocument....,219004,...,2026-05-22T12:58:20Z,<NA>,0,0,None,01NA37BFQNSPXXKLNPUFHI6CY45G7T4DKF,b!FIB3Z0KRfU6tyPRnX3MFBomAIQiIsJtIhXnCtudAYp8r...,ok,2026-06-03T19:39:42+00:00,root/General/7. Credenciamento LPU-BID-RFQ/LPU...
2,expanziobr.sharepoint.com,Comercial,https://expanziobr.sharepoint.com/sites/Comercial,Documents,root/General/Pastas q não pertecem ao comercia...,Análise_de_Maturidade_do_colaborador(a) - Isab...,file,xlsx,application/vnd.openxmlformats-officedocument....,66805,...,2026-05-15T19:50:52Z,<NA>,0,0,None,01NA37BFVG7KUJLXLMW532QV4ELIGYTAN5,b!FIB3Z0KRfU6tyPRnX3MFBomAIQiIsJtIhXnCtudAYp8r...,ok,2026-06-03T19:39:56+00:00,root/General/Pastas q não pertecem ao comercia...
3,expanziobr.sharepoint.com,Comercial,https://expanziobr.sharepoint.com/sites/Comercial,Documents,root/General/Pastas q não pertecem ao comercia...,Planilha de Avaliação de Desempenho Tradiciona...,file,xlsx,application/vnd.openxmlformats-officedocument....,45684,...,2026-05-15T19:49:51Z,<NA>,0,0,None,01NA37BFXU6RIZC6V66B42J3HKWGFSDZPE,b!FIB3Z0KRfU6tyPRnX3MFBomAIQiIsJtIhXnCtudAYp8r...,ok,2026-06-03T19:39:56+00:00,root/General/Pastas q não pertecem ao comercia...
4,expanziobr.sharepoint.com,Comercial,https://expanziobr.sharepoint.com/sites/Comercial,Documents,root/General/Pastas q não pertecem ao comercia...,Ficha de Acompanhamento de Feedback.xlsx,file,xlsx,application/vnd.openxmlformats-officedocument....,17362,...,2026-05-15T19:49:27Z,<NA>,0,0,None,01NA37BFQJ7O7TRHX2YJ5YBZE7RCBBPGZJ,b!FIB3Z0KRfU6tyPRnX3MFBomAIQiIsJtIhXnCtudAYp8r...,ok,2026-06-03T19:39:56+00:00,root/General/Pastas q não pertecem ao comercia...


## Opcional: ler o mesmo Parquet com PySpark

Use esta parte apenas se voce quiser comparar com Spark ou se o volume crescer bastante.

In [ ]:
# from pyspark.sql import SparkSession, functions as F
#
# spark = (
#     SparkSession.builder
#     .appName("analyze-sharepoint-inventory-parquet")
#     .master("local[*]")
#     .getOrCreate()
# )
#
# spark_df = spark.read.parquet(str(PARQUET_PATH))
# spark_df.printSchema()
# spark_df.show(20, truncate=80)